In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro, levene
from scipy.stats import mannwhitneyu
from scipy.stats import chi2_contingency

import warnings
warnings.filterwarnings('ignore')

# Exploring Data

In [2]:
data = pd.read_csv("AB_Test_Results.csv")

data.head()

,USER_ID,VARIANT_NAME,REVENUE
0,737,variant,0.0
1,2423,control,0.0
2,9411,control,0.0
3,7311,control,0.0
4,6174,variant,0.0


In [3]:
data.groupby("VARIANT_NAME").mean()

,USER_ID,REVENUE
VARIANT_NAME,,
control,4989.436798,0.129013
variant,4972.776914,0.070070


In [4]:
data.groupby("VARIANT_NAME").describe()

USER_ID                                                          \
               count         mean          std  min     25%     50%      75%   
VARIANT_NAME                                                                   
control       4984.0  4989.436798  2905.145109  2.0  2466.0  4964.5  7576.25   
variant       5016.0  4972.776914  2876.320625  3.0  2476.5  4958.5  7415.25   

                      REVENUE                                                  
                  max   count      mean       std  min  25%  50%  75%     max  
VARIANT_NAME                                                                   
control       10000.0  4984.0  0.129013  3.007524  0.0  0.0  0.0  0.0  196.01  
variant       10000.0  5016.0  0.070070  1.314802  0.0  0.0  0.0  0.0   58.63

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   USER_ID       10000 non-null  int64  
 1   VARIANT_NAME  10000 non-null  object 
 2   REVENUE       10000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 234.5+ KB


# Normality and Variance Test to Decide Which Method We Will Use

### Shapiro-Wilk Test of Normality Hypotheses

* **$H_0$:** The sample comes from a normally distributed population.
* **$H_1$:** The sample does not come from a normally distributed population.

### Levene's Test for Homogeneity of Variance Hypotheses

* **$H_0$:** The population variances of the groups are equal. ($\sigma_1^2 = \sigma_2^2$)
* **$H_1$:** The population variances of the groups are not equal. ($\sigma_1^2 \neq \sigma_2^2$)

In [6]:
control_group = data[data['VARIANT_NAME'] == 'control']['REVENUE']
variant_group = data[data['VARIANT_NAME'] == 'variant']['REVENUE']

# Shapiro-Wilk Test
stat_control, p_control = shapiro(control_group)
stat_variant, p_variant = shapiro(variant_group)

print(f"Shapiro-Wilk P-Value of Control: {p_control:.4f}")
print(f"Shapiro-Wilk P-Value of Variant: {p_variant:.4f}")

# 2. Levene Test
stat_levene, p_levene = levene(control_group, variant_group)

print(f"Levene Test P-Value: {p_levene:.4f}")

Shapiro-Wilk P-Value of Control: 0.0000
Shapiro-Wilk P-Value of Variant: 0.0000
Levene Test P-Value: 0.2037


### Assumption Checks Results

**1. Normality Check (Shapiro-Wilk Test)**
* **Control Group:** $p$-value = $0.0000$
* **Variant Group:** $p$-value = $0.0000$
* **Conclusion:** Since $p < 0.05$, we reject the null hypothesis ($H_0$). The data for both groups are not normally distributed.

**2. Homogeneity of Variance Check (Levene's Test)**
* **Test Result:** $p$-value = $0.2037$
* **Conclusion:** Since $p > 0.05$, we fail to reject the null hypothesis ($H_0$). The population variances of the two groups are homogeneous.

### Result

We decided to use Mann-Whitney U Test

# A/B Test

In [7]:
stat_mw, p_mw = mannwhitneyu(control_group, variant_group)

print(f"Mann-Whitney U Test P-Value: {p_mw:.4f}")

Mann-Whitney U Test P-Value: 0.4783


### Checks Results

**Mann-Whitney U Test**
* **$p$-value = $0.0000$** 
* **Conclusion:** Since $p > 0.05$, we do not reject the null hypothesis ($H_0$). There are no difference between both groups by Mann-Whitney U Test.

But since our %75 of our groups has 0 (zero) value, this result may lead us wrong decision. 

# A/B Test with Ratio

### Conversion Rate Check (Chi-Square Test of Independence)

* **$H_0$:** Group assignment and conversion status are independent (No difference in conversion rates between groups).
* **$H_1$:** Group assignment and conversion status are dependent (There is a difference in conversion rates between groups).

In [8]:
data['CONVERTED'] = (data['REVENUE'] > 0).astype(int)

contingency_table = pd.crosstab(data['VARIANT_NAME'], data['CONVERTED'])
print("Contingency Table:\n", contingency_table, "\n")

chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-Square Test P-Value: {p_val:.4f}")

Contingency Table:
 CONVERTED        0   1
VARIANT_NAME          
control       4904  80
variant       4944  72 

Chi-Square Test P-Value: 0.5406


**Test Results**
* **$p$-value:** $0.5406$
* **Conclusion:** Since $p > 0.05$, we fail to reject the null hypothesis ($H_0$). Group assignment and conversion are independent. There is no statistically significant difference in conversion rates between the Control and Variant groups.